# model_inference

Loads the overall best model **from the MLflow Model Registry** and predicts the raw, un-preprocessed test set (all preprocessing lives inside the registered pipeline).

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path().resolve()))  # repo root on path

import numpy as np
import pandas as pd
import mlflow

from src.data import load_raw, make_submission
from src.metrics import wmae_report
from src.validation import FOLDS, split_fold
from src.mlflow_utils import setup_mlflow

train, test, features, stores = load_raw()
print(train.shape, test.shape)

from src.postprocess import apply_christmas_shift
from src.mlflow_utils import REGISTRY_MODEL_NAME

setup_mlflow("Inference")

In [ ]:
MODEL_URI = f"models:/{REGISTRY_MODEL_NAME}/latest"   # or /1, /2 ... for a pinned version
model = mlflow.sklearn.load_model(MODEL_URI)
model

In [ ]:
raw_test = test[["Store", "Dept", "Date"]]          # RAW — no preprocessing here
sub = test[["Store", "Dept", "Date"]].copy()
sub["pred"] = model.predict(raw_test)
sub = apply_christmas_shift(sub, pred_col="pred")
make_submission(sub, "pred", "submission.csv")
sub.head()

Upload `submission.csv` to Kaggle:

```
kaggle competitions submit -c walmart-recruiting-store-sales-forecasting -f submission.csv -m "registry model"
```

Record the public/private score in the README comparison table.